# Unsupervised Learning 

This notebook contains the demo-code for: 
+ KMeans Clustering
+ DBSCAN

### Import Packages

In [ ]:
import pandas as pd
import numpy as np 
import kagglehub
import matplotlib.pyplot as plt

from pathlib import Path

### KMeans Clustering

In [ ]:
path = kagglehub.dataset_download("vjchoudhary7/customer-segmentation-tutorial-in-python")
print("Path to dataset files:", path)

data_dir = Path(path)
print(list(data_dir.iterdir()))

file_path = data_dir / "Mall_Customers.csv"

#### Reading and Understanding the Dataset

In [ ]:
df1 = pd.read_csv(file_path)
print(df1.head())

In [ ]:
df1.info()

In [ ]:
print(df1.describe())

Notes: 
+ 4 Variables (str, int, int, int)
+ Each variable looks normally distributed, not skewed

Variables: 
+ Spending Score: Score assigned by the mall based on customer behaviour and spending nature
+ Other variables are self-explanatory

In [ ]:
fig, ax = plt.subplots(figsize = (8,5))

ax.scatter(x=df1['Annual Income (k$)'],
           y=df1['Spending Score (1-100)'],
           zorder = 3)

ax.set(title="Funky Scatter Plot",
       xlabel="Annual Income (k$)",
       ylabel="Spending Score")

ax.grid(alpha = 0.5,
        linestyle ="--",
        zorder=1)

plt.tight_layout()
plt.show()

#### KMeans Demonstration

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

##### Preparing the Data

We want to prepare the data for KMeans clustering. There is not much to do, mainly we should scale the data, convert any type(str) variables into numerical values (only if it makes sense eg. binary, ordinal) and set the KMeans model.

Note: 
+ It is always better to scale your data if your model relies on Euclidean distance (eg. KNN, kmeans). A variable with a large scale can overwhelm other variables on a smaller scale. 

In [ ]:
## Obtain the columns that we want
df1_demo = df1[['Gender', 'Age', 'Annual Income (k$)', 'Spending Score (1-100)']].copy()

## Convert to binary
df1_demo['Gender'] = np.where(df1_demo['Gender'] == 'Male', 1, 0)
df1_demo = df1_demo.rename(columns={'Gender': 'is_male'})

In [ ]:
print(df1_demo.head())

In [ ]:
# If possible, scale the data before using KMeans, as it relies on Euclidean distance
scaler = StandardScaler()
df1_scaled = scaler.fit_transform(df1_demo)

KMeans Parameters (for sklearn):
+ n_clusters: the number of clusters centroids that you desire
+ random_state: ensure replicability
+ n_init: the number of times the algorithm is run with different seeds (hill-climb optimisation)
+ tol: default = 1e-4. The Frobenius norm of the difference in cluster centers of two consecutive iterations to declare convergence. (Think of it as: if the movement in cluster centroid is below `tol`, then we can declare that we have reached the optimal clustering of data points)

In [ ]:
## This step creates an instance of the KMeans model
kmeans = KMeans(n_clusters=5, random_state=42, n_init="auto")
kmeans.fit(df1_scaled)

In [ ]:
print("Cluster Labels:", kmeans.labels_) 

In [ ]:
df1_demo['Label'] = kmeans.labels_
print(df1_demo.head())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

scatter = ax.scatter(
    x=df1_demo['Annual Income (k$)'],
    y=df1_demo['Spending Score (1-100)'],
    c=df1_demo['Label'],
    cmap='viridis',
    zorder=3
)

ax.set(
    title='Customer Segments from KMeans',
    xlabel='Annual Income (k$)',
    ylabel='Spending Score (1-100)'
)
ax.grid(alpha=0.5, linestyle='--', zorder=1)
plt.tight_layout()
plt.show()

Why does it look so bad? 

+ We are projecting multi-dimensional data into a plane 
+ Data are grouped in 'data clouds' in the hyperplane. 
+ We can remove some variables to see better clusters

In [ ]:
print("Centroids:\n", kmeans.cluster_centers_)

In [ ]:
new_customer = pd.DataFrame([{
    'is_male': 1, # male
    'Age': 23, # 23 Years Old
    'Annual Income (k$)': 40, # Annual Income of 40k
    'Spending Score (1-100)': 60
}])

new_scaled = scaler.transform(new_customer)
cluster_label = int(kmeans.predict(new_scaled)[0]) ## this returns a list with 1 element, so select the element with [0] and convert to int

print(f"New Customer Belongs to Cluster {cluster_label}")

new_customer.assign(Cluster=cluster_label)

#### Plotting the Scree Plot

In [ ]:
# Set a range of clusters that you want to test
cluster_range = range(1, 11)
inertias = []

for n in cluster_range:
    model = KMeans(n_clusters=n, random_state=42,n_init='auto')
    model.fit(df1_scaled)
    inertias.append(model.inertia_)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(cluster_range, inertias, marker='o')
ax.set(
    title='Scree Plot: Choosing the Number of Clusters',
    xlabel='Number of clusters (k)',
    ylabel='Inertia'
)
ax.set_xticks(list(cluster_range))
ax.grid(alpha=0.5, linestyle='--')

# Add annotations
ax.annotate("Elbow Point", 
            xy=(5,70), 
            xytext=(6, 150),
            arrowprops={'arrowstyle': '->', 'color':'red'},
            color = 'red')

plt.tight_layout()
plt.show()

##### Better Visualisation

In [ ]:
## Obtain the columns that we want
df1_demo = df1[['Annual Income (k$)', 'Spending Score (1-100)']].copy()

## Scaling
df1_scaled = scaler.fit_transform(df1_demo)

## Fit the KMeans Model
kmeans = KMeans(n_clusters=5, random_state=42, n_init="auto")
kmeans.fit(df1_scaled)

## Get labels
df1_demo['Label'] = kmeans.labels_
print(df1_demo.head())

## Plot the Graph
fig, ax = plt.subplots(figsize=(8, 5))

scatter = ax.scatter(
    x=df1_demo['Annual Income (k$)'],
    y=df1_demo['Spending Score (1-100)'],
    c=df1_demo['Label'],
    cmap='viridis',
    zorder=3
)

ax.set(
    title='Customer Segments from KMeans',
    xlabel='Annual Income (k$)',
    ylabel='Spending Score (1-100)'
)
ax.grid(alpha=0.5, linestyle='--', zorder=1)
plt.tight_layout()
plt.show()

print("Centroids:\n", kmeans.cluster_centers_)

##### Challenge: Try applying DBSCAN to this dataset yourself!

### DBSCAN

In [ ]:
path = kagglehub.dataset_download("berkayalan/sklearn-moons-data-set")
print("Path to dataset files:", path)

data_dir = Path(path)
print(list(data_dir.iterdir()))


file_path = data_dir / "cluster_moons.csv"

#### Reading and Understanding the dataset

In [ ]:
df2 = pd.read_csv(file_path)
print(df2.head())

Notes: This dataset just contains coordinates to half-circles in R2

#### What happens if we just apply KMeans to this?

In [ ]:
df2_kmeans = df2.copy()

## Not scaling here to preserve the shape of the half moons

kmeans = KMeans(n_clusters=2, random_state=42, n_init="auto")
kmeans.fit(df2_kmeans)

df2_kmeans['Label'] = kmeans.labels_
print(df2_kmeans.head())

## Plot the Graph
fig, ax = plt.subplots(figsize=(8, 5))

scatter = ax.scatter(
    x=df2_kmeans['X1'],
    y=df2_kmeans['X2'],
    c=df2_kmeans['Label'],
    cmap='viridis',
    zorder=3
)

ax.set(
    title='Moons Using KMeans',
    xlabel='X1',
    ylabel='X2'
)
ax.grid(alpha=0.5, linestyle='--', zorder=1)
plt.tight_layout()
plt.show()

print("Centroids:\n", kmeans.cluster_centers_)

#### DBSCAN Demonstration

In [ ]:
from sklearn.cluster import DBSCAN

df2_db = df2.copy()

DBSCAN Paramters (for sklearn):
+ eps: (default=0.5) The maximum distance between two samples for one to be considered as in the neighborhood of the other.
+ min_samples: (default=5) The number of data points in a neighborhood for a point to be considered as a core point.
+ metric: (default='euclidean') The metric to use when calculating distance between instances in a feature array.


There are others: 
+ algorithm: {‘auto’, ‘ball_tree’, ‘kd_tree’, ‘brute’}
+ leaf_size: Leaf size passed to BallTree or cKDTree
+ p: The power of the Minkowski metric to be used to calculate distance between points. If None, then p=2 (equivalent to the Euclidean distance).
+ n_jobs: The number of parallel jobs to run.

But these are not as important as the first 3. Feel free to read up on them if you are interested

In [ ]:
# This creates an instance of the DBSCAN model
db = DBSCAN(eps=0.2, min_samples=3, metric='euclidean')
db.fit(df2_db)

In [ ]:
## Similar to KMeans, DBSCAN assigns a cluster label to each data point
labels = db.labels_
df2_db['Label'] = labels # we assign each data point to its cluster

n_clusters = len(set(labels)) - (1 if -1 in labels else 0) ## Calculate the number of unique labels
print("The number of clusters:", n_clusters)

In [ ]:
## Plot the Graph
fig, ax = plt.subplots(figsize=(8, 5))

scatter = ax.scatter(
    x=df2_db['X1'],
    y=df2_db['X2'],
    c=df2_db['Label'],
    cmap='viridis',
    zorder=3
)

ax.set(
    title='Moons Using DBSCAN',
    xlabel='X1',
    ylabel='X2'
)
ax.grid(alpha=0.5, linestyle='--', zorder=1)
plt.tight_layout()
plt.show()

##### Finding Optimal Epsilon With a k-distance Plot

In [ ]:
from sklearn.neighbors import NearestNeighbors

# Use k = 2 * number of dimensions
number_of_dimensions = df2.shape[1]
k = 2 * number_of_dimensions

nearest_neighbors = NearestNeighbors(n_neighbors=k)
nearest_neighbors.fit(df2)
distances, indices = nearest_neighbors.kneighbors(df2)

# The last column contains the distance to each point's kth nearest neighbour.
# The first column contains the point itself
k_distances = np.sort(distances[:, -1]) # [: , -1] translates to "select all rows, from the last (-1) column"

# Plot the k-distance curve
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(0, len(k_distances)), k_distances) # x axis is index 0 to last index of data point
ax.set(
    title=f'{k}-Distance Plot for Choosing DBSCAN eps',
    xlabel='Data points sorted by distance',
    ylabel=f'Distance to {k}th nearest neighbour'
)
ax.grid(alpha=0.5, linestyle='--')

ax.axhline(y=0.05,linestyle = "--", color = "red")
ax.annotate("Optimal eps", xy=(1200, 0.05), xytext=(0, 5), textcoords="offset points", color = "red")
plt.tight_layout()
plt.show()